# Gene Reliability Analysis

This notebook walks through the pyGeneBasis reliability pipeline, which assesses how well each gene in a MERFISH panel faithfully represents the co-expression structure measured in a matched scRNA-seq reference.

**Workflow:**

1. **Compute co-expression matrices** — Pearson correlation between all panel gene pairs, separately in MERFISH and scRNA-seq reference
2. **Score gene reliability** — classify each gene into a failure mode based on how its co-expression structure differs between modalities
3. **Run perturbation analysis** — quantify the impact of each failure mode on cell type mapping accuracy
4. **Visualize results**

---

### Failure modes

| Failure mode | Interpretation |
|---|---|
| `reliable` | Co-expression structure consistent between MERFISH and reference |
| `probe_failure` | Gene is poorly detected in MERFISH (low detection rate vs. reference) |
| `idiosyncratic_noise` | Gene's MERFISH residuals are structured but not aligned with PCA noise |
| `composition_mismatch` | Co-expression differs due to cell type composition differences between datasets |

---

**Prerequisites:**
- MERFISH `.h5ad` with log-normalised counts
- scRNA-seq reference `.h5ad` with log-normalised counts
- A gene panel list (output of `gene_search` or `trim_panel`)
- Cell type labels in both datasets (needed for perturbation analysis)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import pygenebasis as pgb

## 1. Load data

In [ ]:
# ── Edit these paths ──────────────────────────────────────────────────────────
MERFISH_PATH  = "/path/to/merfish.h5ad"       # MERFISH / spatial data
REF_PATH      = "/path/to/reference.h5ad"     # scRNA-seq reference
PANEL_PATH    = "/path/to/gene_panel.csv"      # CSV with a 'gene' column
OUTPUT_DIR    = "/path/to/output"

# Cell type and batch columns in the scRNA-seq reference
# Level keys must be ordered coarsest → finest (e.g. Class → Subclass → Group)
LEVEL_KEYS    = ["Class", "Subclass", "Group"]
BATCH_KEY     = "donor_id"     # set to None for single-batch

# Optional: obs column for stratified subsampling (recommended for large datasets)
CELLTYPE_KEY  = LEVEL_KEYS[-1]  # finest annotation level
LAYER         = None             # expression layer (None → .X)
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
adata_merfish = pgb.read_adata(MERFISH_PATH)
adata_ref     = pgb.read_adata(REF_PATH)
panel_df      = pgb.read_table(PANEL_PATH)
panel_genes   = list(panel_df["gene"])

print(f"MERFISH:    {adata_merfish.n_obs:,} cells × {adata_merfish.n_vars:,} genes")
print(f"Reference:  {adata_ref.n_obs:,} cells × {adata_ref.n_vars:,} genes")
print(f"Panel:      {len(panel_genes)} genes")

# Verify all panel genes are in both datasets
in_merfish = [g for g in panel_genes if g in adata_merfish.var_names]
in_ref     = [g for g in panel_genes if g in adata_ref.var_names]
in_both    = [g for g in panel_genes if g in adata_merfish.var_names and g in adata_ref.var_names]
print(f"Panel genes in MERFISH: {len(in_merfish)}/{len(panel_genes)}")
print(f"Panel genes in ref:     {len(in_ref)}/{len(panel_genes)}")
print(f"Panel genes in both:    {len(in_both)}/{len(panel_genes)}")

## 2. Compute co-expression matrices

`compute_corr_matrices` computes gene × gene Pearson correlation matrices in both the MERFISH and reference datasets. Large datasets can be subsampled for speed.

In [ ]:
corr_data = pgb.compute_corr_matrices(
    adata_merfish,
    adata_ref,
    genes=panel_genes,
    # Optional stratified subsampling — recommended for large datasets
    # merfish_cell_type_key=CELLTYPE_KEY,
    # merfish_n_cells_per_type=200,
    # ref_cell_type_key=CELLTYPE_KEY,
    # ref_n_cells_per_type=200,
    layer=LAYER,
    random_state=0,
)

genes      = corr_data["genes"]
corr_m     = corr_data["merfish_corr"]
corr_r     = corr_data["ref_corr"]
corr_m_rsc = corr_data["merfish_corr_rescaled"]
detect_m   = corr_data["merfish_detect_rate"]
detect_r   = corr_data["ref_detect_rate"]

print(f"Correlation matrices: {corr_m.shape}")

In [ ]:
# Quick look at MERFISH vs reference correlation
off_diag = np.triu(np.ones_like(corr_m, dtype=bool), k=1)
m_vals = corr_m[off_diag]
r_vals = corr_r[off_diag]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(r_vals, m_vals, s=0.3, alpha=0.4, rasterized=True)
lim = max(abs(r_vals).max(), abs(m_vals).max()) * 1.05
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.axline((0, 0), slope=1, color="red", lw=0.8, ls="--")
ax.set_xlabel("Reference co-expression (Pearson r)")
ax.set_ylabel("MERFISH co-expression (Pearson r)")
ax.set_title("Co-expression agreement (gene pairs)")
fig.tight_layout()
plt.show()

In [ ]:
# Detection rate comparison
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(detect_r, detect_m, s=15, alpha=0.6)
ax.axline((0, 0), slope=1, color="red", lw=0.8, ls="--")
ax.set_xlabel("Reference detection rate")
ax.set_ylabel("MERFISH detection rate")
ax.set_title("Detection rate comparison")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.tight_layout()
plt.show()

## 3. Score gene reliability

`compute_gene_reliability` (convenience wrapper) computes correlation matrices and classifies each gene into a failure mode.

**PC selection:** Parallel analysis (Horn's method) is used by default — column-wise permutations define a null eigenvalue distribution, and the number of PCs is the first rank where the real eigenvalue falls below the 95th-percentile null. This is parameter-free and more principled than a fixed variance threshold.

Alternatively, use the two-step form if you want to inspect the correlation matrices first (as shown above).

In [ ]:
# Two-step form: score from pre-computed corr_data
results, thresholds = pgb.score_gene_reliability(
    corr_data,
    pca_method="parallel_analysis",  # recommended
    n_permutations=100,
    pa_percentile=95.0,
    n_jobs=-1,
)

print(f"\nFailure mode breakdown ({len(results)} genes):")
print(results["failure_mode"].value_counts().to_string())

In [ ]:
# Save reliability scores
pgb.write_csv(results, f"{OUTPUT_DIR}/reliability_scores.csv")
print(results.sort_values("failure_mode").head(20).to_string())

### PC selection diagnostics

In [ ]:
# Scree plot with parallel analysis null threshold
pca_info    = thresholds["_pca"]
real_eig    = pca_info["real_eigenvalues"]
null_thresh = pca_info["null_threshold"]
evr         = pca_info["explained_variance_ratio"]
n_pcs_used  = thresholds["n_pcs"]

ranks = np.arange(1, len(real_eig) + 1)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

ax1.bar(ranks, real_eig, color="steelblue", alpha=0.7, label="Real eigenvalue")
ax1.plot(ranks, null_thresh, color="red", lw=1.5, label="95th-pct null (parallel analysis)")
ax1.axvline(n_pcs_used + 0.5, color="black", lw=1.2, ls="--", label=f"k = {n_pcs_used}")

cum_var = np.cumsum(evr) * 100
ax2.plot(ranks, cum_var, color="orange", lw=1.5, ls=":", label="Cumulative variance (%)")
ax2.set_ylabel("Cumulative variance (%)", color="orange")
ax2.tick_params(axis="y", colors="orange")

ax1.set_xlabel("PC rank")
ax1.set_ylabel("Eigenvalue")
ax1.set_title(f"PC selection: {n_pcs_used} PCs chosen by parallel analysis")

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

### Reliability score distributions

In [ ]:
fig, axes = pgb.plot_metric_distributions(results, thresholds)
plt.show()

In [ ]:
# Failure mode scatter (fidelity vs idiosyncratic noise)
fig, ax = pgb.plot_failure_mode_scatter(results)
plt.show()

## 4. Perturbation analysis

`run_perturbation_analysis` quantifies the impact of each failure mode on cell type mapping accuracy in the scRNA-seq reference.

For each failure mode:
- **Perturb**: add failure-mode-specific noise to affected genes, rebuild kNN graph, measure drop in mapping accuracy (delta CT)
- **Remove**: exclude affected genes from panel entirely, measure delta CT
- **Removal benefit**: `mean(delta_CT_perturbed) − delta_CT_removal` — positive means removing is better than keeping broken genes

**Two voting strategies** are available:
- `"unconstrained"` — all k neighbours vote (matches `get_celltype_mapping`)
- `"constrained"` — only neighbours sharing the same parent-level label vote (prevents cross-lineage contamination)
- `"both"` (default) — runs both strategies from a single kNN build

In [ ]:
# Subset reference to panel genes for perturbation analysis
adata_panel = adata_ref[:, [g for g in panel_genes if g in adata_ref.var_names]].copy()

perturb_results = pgb.run_perturbation_analysis(
    adata_panel,
    results,
    level_keys=LEVEL_KEYS,
    batch_key=BATCH_KEY,
    n_neighbors=5,
    knn_method="approx",
    n_replicates=5,
    n_cells_per_group=5000,
    vote="both",
    verbose=True,
)

In [ ]:
# Save perturbation results
for key, df in perturb_results.items():
    pgb.write_csv(df, f"{OUTPUT_DIR}/perturb_{key}.csv")
    print(f"  {key}: {df.shape}")

print("\nSummary (mean delta CT by failure mode and level):")
print(perturb_results["summary"].to_string(index=False))

### Perturbation summary plot

In [ ]:
fig, axes = pgb.plot_perturbation_summary(
    perturb_results["summary"],
    title="Mean drop in cell type mapping accuracy by failure mode",
)
plt.show()

### Delta CT heatmaps (per annotation level)

In [ ]:
for level in LEVEL_KEYS:
    fig, ax = pgb.plot_delta_ct_heatmap(
        perturb_results["delta_ct"],
        level=level,
        title=f"Delta CT heatmap — {level}",
    )
    plt.show()

### Removal benefit

In [ ]:
for level in LEVEL_KEYS:
    fig, ax = pgb.plot_removal_benefit(
        perturb_results["removal_benefit"],
        level=level,
        title=f"Removal benefit — {level}\n"
              "(positive = removing unreliable genes helps more than keeping them)",
    )
    plt.show()

### Constrained vote results (hierarchically-aware accuracy)

In [ ]:
if "summary_constrained" in perturb_results:
    print("Summary (constrained vote):")
    print(perturb_results["summary_constrained"].to_string(index=False))

    fig, axes = pgb.plot_perturbation_summary(
        perturb_results["summary_constrained"],
        title="Mean delta CT (constrained vote)",
    )
    plt.show()

---

## CLI equivalent

See `docs/scripts/run_reliability.sh` for a complete example.

```bash
# Score co-expression reliability (MERFISH vs scRNA-seq)
pygenebasis reliability score-coexp \
    --merfish merfish.h5ad \
    --ref reference.h5ad \
    --panel gene_panel.csv \
    --output results/reliability_scores.csv

# Run perturbation analysis
pygenebasis reliability perturb \
    --adata reference.h5ad \
    --panel gene_panel.csv \
    --results results/reliability_scores.csv \
    --celltype-key Group \
    --output results/perturbation_results.csv
```